In [ ]:
# We import the neccessary packages in the beginning
import warnings
import os
import math
from statistics import mean,stdev
import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.conversion.bpmn import converter as bpmn_converter
from sklearn.impute import SimpleImputer
import copy
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler    
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import sklearn
import tqdm
import time
import ebi
warnings.filterwarnings('ignore')


In [ ]:
import json
import re
import pandas as pd
from datetime import datetime, timezone, timedelta


def _parse_exs_timestamp(raw):
    """
    Parse timestamps of the form "2017-05-09 00:00:00 +01:00".
    Python's fromisoformat requires the offset directly after the time
    (no space before ±), so we strip the extra space.
    Returns a timezone-aware datetime, or None if raw is None/empty.
    """
    if not raw:
        return None
    fixed = re.sub(r'\s+([+-]\d{2}:\d{2})$', r'\1', raw.strip())
    return datetime.fromisoformat(fixed)


def _parse_utilisation(raw, index):
    """
    Parse resource_utilisation: accepts a float, an int, or a fraction string
    like '1/9'. Returns a float in [0, 1] or None.
    """
    if raw is None:
        return None
    try:
        util = float(raw)
    except (ValueError, TypeError):
        # try "numerator/denominator" format
        s = str(raw).strip()
        if '/' in s:
            parts = s.split('/', 1)
            try:
                util = float(parts[0]) / float(parts[1])
            except (ValueError, ZeroDivisionError):
                raise ValueError(f"Execution #{index}: unparseable resource_utilisation={raw!r}")
        else:
            raise ValueError(f"Execution #{index}: unparseable resource_utilisation={raw!r}")
    if not (0.0 <= util <= 1.0):
        raise ValueError(f"Execution #{index}: resource_utilisation={util} out of [0,1]")
    return util


def parse_exs(path):
    """
    Parse a .exs execution replay file into a DataFrame.

    .exs files are produced by replaying a Petri net against an event log using
    pm4py's stochastic replay.  Each entry in "executions" records one transition
    firing: which transition fired, what else was simultaneously enabled, and the
    resource utilisation values at the moment of firing.

    ── Format detection ────────────────────────────────────────────────────────
    Two formats are distinguished by the presence of the key
    "resource_utilisation_fired_transition" in the first entry:

    Old format  – single utilisation scalar per firing:
        resource_utilisation   float or fraction string (e.g. "1/9")

    New format (alt-util)  – per-transition utilisation at each choice point:
        fired_transition                              int  — Petri net transition id that fired
        trace                                         int  — case/trace id
        activity                                      str|null  — activity label; null = silent (τ)
        also_in_log                                   bool
        move_index                                    int  — ordinal position of this firing
                                                             in the Petri net replay for this trace
        move_index_of_enablement                      int|null
        other_enabled_transitions                     list[int]  — transition ids that were also
                                                                    enabled when this one fired
        resource_utilisation_fired_transition         str|float|null
        other_enabled_transitions_resource_utilisation list[str|float|null]  — parallel to
                                                                    other_enabled_transitions
        resource                                      str|null
        time_of_enablement / time_of_execution        str|null  — ISO 8601 timestamps

    ── Resource utilisation values ─────────────────────────────────────────────
    Utilisation is normalised to [0, 1].  The raw value can be:
      • a float already in [0, 1]
      • a fraction string such as "1/9" (1 active case / 9 total capacity)
      • null  — the transition has no resource assigned (always the case for
                 silent transitions and for model-move firings that the aligner
                 inserted without a matching log event)

    ── alt_utils: the key field for causal analysis ────────────────────────────
    In the new format, every choice-point row carries two utilisation values:

      resource_utilisation_fired_transition
          The load on the resource assigned to the transition that ACTUALLY fired.
          This becomes the treatment variable T in the causal models.

      other_enabled_transitions_resource_utilisation
          A parallel list aligned with other_enabled_transitions: the load on the
          resource assigned to each ALTERNATIVE transition at the moment the choice
          was made.  Together with the fired utilisation, these form the confounder
          set X (all resource loads observable at the decision point).

    alt_utils is the parsed version of the latter: a dict {tid: float|None}.
    Alternatives without a resource (silent transitions or unassigned) have
    util=None and are excluded from downstream analysis.

    ── move_index and move_index_of_enablement ─────────────────────────────────
    move_index
        A per-trace ordinal assigned by the Petri net aligner.  Not necessarily
        consecutive — gaps occur when the aligner performs internal steps not
        recorded in this file (e.g. invisible transitions in a DFG-derived net
        that are absorbed into alignment moves and not surfaced as separate rows).

    move_index_of_enablement
        The move_index of the firing that produced the last token this transition
        needed to become enabled.  Concretely: if transition A fired at move_index=k
        and thereby put a token on the place that enabled B, then B's
        move_index_of_enablement = k, regardless of how many steps elapsed between
        A's firing and B's actual firing.
        None for transitions enabled by the initial marking (no predecessor).
        Used by visualize_dfg / build_petri_net_graph to reconstruct arc structure.

    ── also_in_log and model moves ─────────────────────────────────────────────
    The Petri net aligner may insert "model moves": transition firings required by
    the model that have no counterpart in the real event log.  These have
    also_in_log=False and resource_utilisation=None (no real resource was used).
    They appear in the replay to bridge structural gaps in a DFG-derived Petri net
    and should generally be ignored for causal analysis — extract_decision_points
    already excludes them because their utilisation is None.

    ── is_decision_point ───────────────────────────────────────────────────────
    True when ALL of the following hold:
      • the transition is visible (activity is not None)
      • at least one other visible transition was simultaneously enabled
    These rows are the XOR gateway decision points used by extract_decision_points
    and all downstream causal analysis functions.

    ── Output columns ───────────────────────────────────────────────────────────
        case                      int32  — trace id
        activity                  category — activity label (None for silent)
        transition                int32  — Petri net transition id
        other_enabled             list[int]
        choice_set                list[int]  — sorted(fired ∪ other_enabled)
        n_alternatives            int  — len(choice_set)
        is_decision_point         bool
        is_silent                 bool
        resource                  category
        resource_utilisation      float|None  — treatment T
        also_in_log               bool
        time_of_enablement        datetime (UTC-aware)|NaT
        time_of_execution         datetime (UTC-aware)|NaT
        alt_utils  (new only)     dict {tid: float|None}  — alternative utils
        move_index (new only)     int|None
        move_index_of_enablement  (new only) int|None
    """
    with open(path) as f:
        data = json.load(f)

    if "executions" not in data:
        raise ValueError(f"No 'executions' key found in {path}")

    # Detect format: new files have resource_utilisation_fired_transition
    sample = data["executions"][0] if data["executions"] else {}
    is_new_format = "resource_utilisation_fired_transition" in sample

    rows = []
    for i, e in enumerate(data["executions"]):

        # ── Utilisation of the fired transition ───────────────────────────────
        if is_new_format:
            util_raw = e.get("resource_utilisation_fired_transition")
        else:
            util_raw = e.get("resource_utilisation")
        util = _parse_utilisation(util_raw, i)

        # ── Choice set ────────────────────────────────────────────────────────
        fired      = int(e["fired_transition"])
        others     = list(e.get("other_enabled_transitions") or [])
        choice_set = sorted(set([fired] + others))

        # ── Transition type flags ─────────────────────────────────────────────
        activity  = e.get("activity")
        is_silent = activity is None

        is_decision_point = (not is_silent) and (len(others) > 0)

        # ── Silent transition consistency check ───────────────────────────────
        if is_silent:
            if e.get("resource") is not None:
                raise ValueError(f"Execution #{i}: silent transition has a resource set")
            if util is not None:
                raise ValueError(f"Execution #{i}: silent transition has resource_utilisation set")

        # ── Alternative utilisation (new format only) ─────────────────────────
        # All non-silent enabled transitions carry a util value by design.
        alt_utils = {}
        if is_new_format:
            alt_utils_raw = e.get("other_enabled_transitions_resource_utilisation") or []
            for t_id, u_raw in zip(others, alt_utils_raw):
                alt_utils[int(t_id)] = _parse_utilisation(u_raw, i)

        row = {
            "case":                 int(e["trace"]),
            "activity":             activity,
            "transition":           fired,
            "other_enabled":        others,
            "choice_set":           choice_set,
            "n_alternatives":       len(choice_set),
            "is_decision_point":    is_decision_point,
            "is_silent":            is_silent,
            "resource":             e.get("resource"),
            "resource_utilisation": util,
            "also_in_log":          bool(e.get("also_in_log", False)),
            "time_of_enablement":   _parse_exs_timestamp(e.get("time_of_enablement")),
            "time_of_execution":    _parse_exs_timestamp(e.get("time_of_execution")),
        }
        if is_new_format:
            row["alt_utils"] = alt_utils
            row["move_index"] = e.get("move_index")
            row["move_index_of_enablement"] = e.get("move_index_of_enablement")

        rows.append(row)

    df = pd.DataFrame(rows)

    # ── Post-parse dtype enforcement ──────────────────────────────────────────
    df["case"]       = df["case"].astype("int32")
    df["transition"] = df["transition"].astype("int32")

    df["activity"]  = df["activity"].astype("category")
    df["resource"]  = df["resource"].astype("category")

    # normalize datetime columns to UTC so .dt accessor always works,
    # even when the source file has mixed UTC offsets (e.g. DST transitions)
    for _col in ["time_of_execution", "time_of_enablement"]:
        df[_col] = pd.to_datetime(df[_col], utc=True, errors="coerce")

    return df

import os

def list_files(base_path, printing=True):
    """Return and print a list of files in the directory with indices."""
    try:
        files = [f for f in os.listdir(base_path)
                 if os.path.isfile(os.path.join(base_path, f))]
        files.sort()
    except FileNotFoundError:
        print(f"Directory not found: {base_path}")
        return []

    if not files:
        print("No files found.")
        return []

    if printing:
        for i, f in enumerate(files):
            print(f"{i}: {f}")

    return files


def get_file_by_index(base_path, index, printing=True):
    files = list_files(base_path, printing)

    if not files:
        raise ValueError("No files available.")

    if index < 0 or index >= len(files):
        raise IndexError(f"Index {index} out of range (0–{len(files)-1}).")

    return os.path.join(base_path, files[index])





In [ ]:
import importlib, causal_analysis, reporting
importlib.reload(causal_analysis)
importlib.reload(reporting)
import csv
from causal_analysis import (
    extract_decision_points, get_feature_cols, DEFAULT_FEATURES_BASE,
    double_ml_test, causal_forest_dml, backdoor_check,
)
from reporting import generate_report

MINERS         = ["dfg-miner", "flw-miner", "im-miner", "imf-miner"]
INPUT_BASE     = "input/3-executions-per-miner"
OUTPUT_BASE    = "output"
null_util_fill = "mean"

In [ ]:
# ── Pick a single file ────────────────────────────────────────────────────────
miner_i        = "imf-miner"
null_util_fill = "mean"

input_dir  = os.path.join(INPUT_BASE, miner_i)
output_dir = os.path.join(OUTPUT_BASE, miner_i)

file_path = get_file_by_index(input_dir, 5)
file_path

In [ ]:
file_name  = os.path.splitext(os.path.basename(file_path))[0]
_time_file = file_path + ".time"
t_replay   = float(open(_time_file).read().strip()) if os.path.exists(_time_file) else None

_t0 = time.perf_counter()
df  = parse_exs(file_path)
t_parse = time.perf_counter() - _t0

_t0 = time.perf_counter()
decisions = extract_decision_points(df)
t_extract = time.perf_counter() - _t0

FEATURES = get_feature_cols(decisions)
_n_base     = len(DEFAULT_FEATURES_BASE)
_n_res      = sum(c.startswith("res_")      for c in FEATURES)
_n_alt_util = sum(c.startswith("alt_util_") for c in FEATURES)

print("INPUT:")
print(f"  file           = {file_name}")
print(f"  miner          = {miner_i}")
print(f"  null_util_fill = {null_util_fill!r}")
print(f"  n_cases        = {df['case'].nunique():,}")
print(f"  n_executions   = {len(df):,}")
print(f"  n_decision_rows= {len(decisions):,}")
print(f"  n_choice_sets  = {decisions['choice_set'].nunique() if len(decisions) else 0}")
print(f"  n_confounders  = {len(FEATURES)}  ({_n_base} base + {_n_res} res + {_n_alt_util} alt_util)")
print(f"  T  = resource_utilisation  (continuous ∈ [0,1])")
print()
display(decisions[["choice_set", "chosen", "util"] + FEATURES[:6]].head(6))

print("\n── STEP 1: Double ML ──")
_t0 = time.perf_counter()
dml_results = double_ml_test(decisions, FEATURES, null_util_fill=null_util_fill)
t_dml = time.perf_counter() - _t0
display(dml_results)

print("\n── STEP 2: Causal Forest DML ──")
_t0 = time.perf_counter()
summary, artifacts = causal_forest_dml(decisions, FEATURES, verbose=False, null_util_fill=null_util_fill)
t_forest = time.perf_counter() - _t0
display(summary)

print("\n── STEP 3: Backdoor adequacy ──")
_t0 = time.perf_counter()
backdoor = backdoor_check(decisions, FEATURES, null_util_fill=null_util_fill)
t_backdoor = time.perf_counter() - _t0
display(backdoor)

os.makedirs(output_dir, exist_ok=True)
paths = generate_report(
    decisions   = decisions,
    lr_results  = pd.DataFrame(),
    dml_results = dml_results,
    summary     = summary,
    artifacts   = artifacts,
    dr          = pd.DataFrame(),
    backdoor    = backdoor,
    output_dir  = os.path.join(output_dir, file_name),
    file_stem   = file_name,
)
print(f"\nExcel : {paths['excel']}")
print(f"PDF   : {paths['pdf']}")

n_significant = int((summary["reject_H0"] == True).sum()) if not summary.empty else 0
_row = {
    "file":                 file_name,
    "null_util_fill":       null_util_fill,
    "n_cases":              df["case"].nunique(),
    "n_executions":         len(df),
    "n_decision_rows":      len(decisions),
    "n_choice_sets":        decisions["choice_set"].nunique() if len(decisions) else 0,
    "n_transitions_fitted": len(artifacts),
    "n_significant":        n_significant,
    "t_replay_s":           round(t_replay, 3) if t_replay is not None else None,
    "t_parse_s":            round(t_parse,   3),
    "t_extract_s":          round(t_extract, 3),
    "t_lr_s":               None,
    "t_dml_s":              round(t_dml,     3),
    "t_forest_s":           round(t_forest,  3),
    "t_backdoor_s":         round(t_backdoor,3),
    "t_pipeline_s":         round(t_parse + t_extract + t_dml + t_forest + t_backdoor, 3),
}
print("\n── Timing & descriptives ──")
display(pd.DataFrame([_row]))

_csv = os.path.join(OUTPUT_BASE, "feasibility_study.csv")
_write_header = not os.path.exists(_csv)
with open(_csv, "a", newline="") as _f:
    _w = csv.DictWriter(_f, fieldnames=list(_row.keys()))
    if _write_header:
        _w.writeheader()
    _w.writerow(_row)

In [ ]:
df

In [ ]:
df_i=df[df['case']==4]
df_i

In [ ]:
importlib.reload(causal_analysis)
from causal_analysis import extract_decision_points, get_feature_cols, DEFAULT_FEATURES_BASE

rows = []
for _miner in MINERS:
    _input_dir = os.path.join(INPUT_BASE, _miner)
    _exs_files = sorted(
        os.path.join(_input_dir, f)
        for f in os.listdir(_input_dir)
        if f.endswith(".exs")
    )
    for _fp in _exs_files:
        _df        = parse_exs(_fp)
        _decisions = extract_decision_points(_df)
        _feat      = get_feature_cols(_decisions)

        _n_base     = len(DEFAULT_FEATURES_BASE)
        _n_res      = sum(c.startswith("res_")      for c in _feat)
        _n_alt_util = sum(c.startswith("alt_util_") for c in _feat)
        _n_total    = len(_feat)

        rows.append({
            "miner":      _miner,
            "file":       os.path.basename(_fp),
            "n_base":     _n_base,
            "n_res":      _n_res,
            "n_alt_util": _n_alt_util,
            "n_total":    _n_total,
        })
        print(f"[{_miner}] {os.path.basename(_fp)}: "
              f"{_n_base} base + {_n_res} res + {_n_alt_util} alt_util = {_n_total}")

confounder_summary = pd.DataFrame(rows)
display(confounder_summary)

In [ ]:
confounder_summary.to_excel("confounders.xlsx")

In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

CONFOUNDER_TIMEOUT_S = 300  # 5 minutes per step per log

def _run_with_timeout(fn, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as _pool:
        return _pool.submit(fn, *args, **kwargs).result(timeout=CONFOUNDER_TIMEOUT_S)

csv_path = os.path.join(OUTPUT_BASE, "feasibility_study.csv")

for miner in MINERS:
    input_dir  = os.path.join(INPUT_BASE, miner)
    output_dir = os.path.join(OUTPUT_BASE, miner)
    os.makedirs(output_dir, exist_ok=True)

    exs_files = sorted(
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(".exs")
    )

    print(f"\n{'='*60}")
    print(f"MINER: {miner}  ({len(exs_files)} files)")
    print(f"{'='*60}")

    for file_path in exs_files:
        file_name = os.path.splitext(os.path.basename(file_path))[0]

        _time_file = file_path + ".time"
        t_replay = float(open(_time_file).read().strip()) if os.path.exists(_time_file) else None

        _t0 = time.perf_counter()
        df = parse_exs(file_path)
        t_parse = time.perf_counter() - _t0

        _t0 = time.perf_counter()
        decisions = extract_decision_points(df)
        t_extract = time.perf_counter() - _t0

        if decisions.empty:
            print(f"\n── {file_name} — no decision points, skipping.")
            continue

        FEATURES    = get_feature_cols(decisions)
        _n_base     = len(DEFAULT_FEATURES_BASE)
        _n_res      = sum(c.startswith("res_")      for c in FEATURES)
        _n_alt_util = sum(c.startswith("alt_util_") for c in FEATURES)

        print(f"\n── {file_name} ──")
        print("INPUT:")
        print(f"  file           = {file_name}")
        print(f"  miner          = {miner}")
        print(f"  null_util_fill = {null_util_fill!r}")
        print(f"  n_cases        = {df['case'].nunique():,}")
        print(f"  n_executions   = {len(df):,}")
        print(f"  n_decision_rows= {len(decisions):,}")
        print(f"  n_choice_sets  = {decisions['choice_set'].nunique()}")
        print(f"  n_confounders  = {len(FEATURES)}  ({_n_base} base + {_n_res} res + {_n_alt_util} alt_util)")
        print(f"  T  = resource_utilisation  (continuous ∈ [0,1])")
        print()
        display(decisions[["choice_set", "chosen", "util"] + FEATURES[:6]].head(6))

        print("\n── STEP 1: Double ML ──")
        _t0 = time.perf_counter()
        try:
            dml_results   = _run_with_timeout(double_ml_test, decisions, FEATURES, null_util_fill=null_util_fill)
            t_dml_timeout = False
        except FuturesTimeoutError:
            dml_results   = pd.DataFrame()
            t_dml_timeout = True
            print(f"  [TIMEOUT] double_ml_test exceeded {CONFOUNDER_TIMEOUT_S}s — skipped")
        t_dml = time.perf_counter() - _t0
        if not dml_results.empty:
            display(dml_results)

        print("\n── STEP 2: Causal Forest DML ──")
        _t0 = time.perf_counter()
        try:
            summary, artifacts = _run_with_timeout(causal_forest_dml, decisions, FEATURES, verbose=False, null_util_fill=null_util_fill)
            t_forest_timeout   = False
        except FuturesTimeoutError:
            summary, artifacts = pd.DataFrame(), {}
            t_forest_timeout   = True
            print(f"  [TIMEOUT] causal_forest_dml exceeded {CONFOUNDER_TIMEOUT_S}s — skipped")
        t_forest = time.perf_counter() - _t0
        if not summary.empty:
            display(summary)

        print("\n── STEP 3: Backdoor adequacy ──")
        _t0 = time.perf_counter()
        try:
            backdoor           = _run_with_timeout(backdoor_check, decisions, FEATURES, null_util_fill=null_util_fill)
            t_backdoor_timeout = False
        except FuturesTimeoutError:
            backdoor           = pd.DataFrame()
            t_backdoor_timeout = True
            print(f"  [TIMEOUT] backdoor_check exceeded {CONFOUNDER_TIMEOUT_S}s — skipped")
        t_backdoor = time.perf_counter() - _t0
        if not backdoor.empty:
            display(backdoor)

        file_output_dir = os.path.join(output_dir, file_name)
        paths = generate_report(
            decisions   = decisions,
            lr_results  = pd.DataFrame(),
            dml_results = dml_results,
            summary     = summary,
            artifacts   = artifacts,
            dr          = pd.DataFrame(),
            backdoor    = backdoor,
            output_dir  = file_output_dir,
            file_stem   = file_name,
        )
        print(f"  Excel : {paths['excel']}")
        print(f"  PDF   : {paths['pdf']}")

        n_significant = int((summary["reject_H0"] == True).sum()) if not summary.empty else 0
        _row = {
            "file":                 file_name,
            "null_util_fill":       null_util_fill,
            "n_cases":              df["case"].nunique(),
            "n_executions":         len(df),
            "n_decision_rows":      len(decisions),
            "n_choice_sets":        decisions["choice_set"].nunique(),
            "n_transitions_fitted": len(artifacts),
            "n_significant":        n_significant,
            "t_replay_s":           round(t_replay,   3) if t_replay is not None else None,
            "t_parse_s":            round(t_parse,    3),
            "t_extract_s":          round(t_extract,  3),
            "t_lr_s":               None,
            "t_dml_s":              round(t_dml,      3) if not t_dml_timeout     else None,
            "t_forest_s":           round(t_forest,   3) if not t_forest_timeout  else None,
            "t_backdoor_s":         round(t_backdoor, 3) if not t_backdoor_timeout else None,
            "t_pipeline_s":         round(t_parse + t_extract + t_dml + t_forest + t_backdoor, 3),
            "timed_out":            t_dml_timeout or t_forest_timeout or t_backdoor_timeout,
        }
        print("\n── Timing & descriptives ──")
        display(pd.DataFrame([_row]))

        write_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as _f:
            w = csv.DictWriter(_f, fieldnames=list(_row.keys()))
            if write_header:
                w.writeheader()
            w.writerow(_row)

print(f"\nDone. Results saved to {csv_path}")
